# Phase 2 - MLflow experiment tracking and model registry


In [ ]:
from pathlib import Path
import json
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

root = Path('.').resolve()
tracking_dir = root / 'mlruns'
mlflow.set_tracking_uri(f'file://{tracking_dir}')
mlflow.set_experiment('wine-quality-experiment')

df = pd.read_csv(root / 'data' / 'winequality-red.csv')
features = [c for c in df.columns if c not in {'quality', 'Id'}]
X, y = df[features], df['quality']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

models = {
    'linear_regression': LinearRegression(),
    'random_forest': RandomForestRegressor(n_estimators=100, random_state=42),
}
results = []
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, pred)
        rmse = root_mean_squared_error(y_test, pred)
        r2 = r2_score(y_test, pred)

        if hasattr(model, 'get_params'):
            mlflow.log_params(model.get_params())
        mlflow.log_metric('mae', float(mae))
        mlflow.log_metric('rmse', float(rmse))
        mlflow.log_metric('r2', float(r2))
        mlflow.set_tags({'algorithm': name, 'dataset_version': 'v1'})

        residuals = (y_test - pred)
        plt.figure(figsize=(6, 4))
        plt.hist(residuals, bins=10)
        plt.title(f'Error distribution - {name}')
        plt.xlabel('Error')
        plt.ylabel('Frequency')
        plot_path = root / f'error_distribution_{name}.png'
        plt.tight_layout()
        plt.savefig(plot_path)
        plt.close()
        mlflow.log_artifact(str(plot_path), artifact_path='plots')

        alpha = 0.05
        lr_sm = sm.OLS(y_train, sm.add_constant(X_train)).fit()
        conf_interval = lr_sm.conf_int(alpha)
        ci_path = root / f'confidence_interval_{name}.csv'
        conf_interval.to_csv(ci_path)
        mlflow.log_artifact(str(ci_path), artifact_path='confidence_intervals')

        mlflow.sklearn.log_model(model, artifact_path='model')
        results.append({'name': name, 'run_id': mlflow.active_run().info.run_id, 'rmse': rmse})

best = sorted(results, key=lambda x: x['rmse'])[0]
model_uri = f"runs:/{best['run_id']}/model"
registered = mlflow.register_model(model_uri=model_uri, name='wine-quality-model')
best


Use `mlflow ui` locally to compare runs and include screenshots in `phase2.pdf`.
